# EDA — Price Action

Basit keşifsel analiz şablonu. Trade journal'dan veri çek, KPI'ları çıkar,
rejim bazlı dağılım/equity curve üret. Synthetic data ile çalışır; gerçek
Postgres mevcutsa otomatik bağlanır.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from price_action.analytics.kpi import compute_kpis, attribution
from price_action.analytics.bias_checks import check_concentration, check_recency
from scripts.seed_data import generate_trades, generate_ohlcv

In [ ]:
# 1. Trade verisi yükle (gerçek journal varsa kullan, yoksa synthetic)
try:
    from price_action.analytics.journal import Journal
    j = Journal()
    trades = j.query_trades(limit=500)
    if trades.empty:
        trades = generate_trades(150)
except Exception:
    trades = generate_trades(150)
trades.head()

In [ ]:
# 2. KPI snapshot
kpis = compute_kpis(trades)
pd.Series(kpis).to_frame('value')

In [ ]:
# 3. Equity curve
df = trades.sort_values('exit_ts').copy()
df['equity'] = 10_000.0 + df['realized_pnl_usdt'].cumsum()
df.set_index('exit_ts')['equity'].plot(title='Equity (USDT)', figsize=(10, 4))
plt.grid(True); plt.show()

In [ ]:
# 4. Atribüsyon — strateji & pattern
display(attribution(trades, by='strategy'))
display(attribution(trades, by='pattern'))

In [ ]:
# 5. Bias checks
print(check_concentration(trades))
print(check_recency(trades))